# Asesinato en SQL City

<div align="center">
<img src="./img/murder.png" width="800" height="600">
</div>

A crime has taken place and the detective needs your help. The detective gave you the crime scene report, but you somehow lost it. You vaguely remember that the crime was a ​murder​ that occurred sometime on ​Jan.15, 2018​ and that it took place in ​SQL City​. Start by retrieving the corresponding crime scene report from the police department’s database.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
# Conectamos con la base de datos cervezas.db
connection = sqlite3.connect('data/sql-murder-mystery.db')

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

Echamos un primer vistazo a las tablas


In [4]:
query = '''
select name 
    from sqlite_master
    where type = 'table'
'''
sql_query(query)


,name
0,crime_scene_report
1,drivers_license
2,person
3,facebook_event_checkin
4,interview
5,get_fit_now_member
6,get_fit_now_check_in
7,income
8,solution


In [5]:
query = 'select * from crime_scene_report'
sql_query(query)


,date,type,description,city
0,20180115,robbery,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,murder,Life? Dont talk to me about life.,Albany
2,20180115,murder,"Mama, I killed a man, put a gun against his he...",Reno
3,20180215,murder,REDACTED REDACTED REDACTED,SQL City
4,20180215,murder,Someone killed the guard! He took an arrow to ...,SQL City
...,...,...,...,...
1223,20180430,bribery,\n,Garden Grove
1224,20180430,fraud,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,assault,\n,Fontana
1226,20180501,assault,be NO mistake about it: it was neither more no...,Trenton


Hay muchos casos de crímenes, el nuestro se trata de un asesinato en SQL City el día 2018/01/15

In [6]:
query = '''
select date as Fecha, type as Tipo, city as Ciudad, description as Detalles 
    from crime_scene_report
    where type = 'murder'
      and date = '20180115'
      and city = 'SQL City';
'''
murderSQL = sql_query(query)
murderSQL

,Fecha,Tipo,Ciudad,Detalles
0,20180115,murder,SQL City,Security footage shows that there were 2 witne...


In [7]:
print(*murderSQL['Detalles'])

Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".


Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".

Información clave:
- Testigo1: vive en la última casa de Northwestern Dr
- Testigo2: Annabel, vive en Franklin Ave

In [8]:
query = '''
select id, name, license_id, max(address_number), address_street_name, ssn
    from person
    where address_street_name = 'Northwestern Dr'
'''
testigo1 = sql_query(query)
testigo1

,id,name,license_id,max(address_number),address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


In [9]:
query = '''
select id, name, license_id, address_number, address_street_name, ssn
    from person
    where name like '%Annabel%'
    and address_street_name = 'Franklin Ave'
'''
testigo2 = sql_query(query)
testigo2

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


Leemos las declaraciones de los testigos

In [10]:
query = '''
select person_id, name, transcript
    from interview
        inner join person on id = person_id
    where person_id in (14887,16371)
'''
declaraciones = sql_query(query)
declaraciones

,person_id,name,transcript
0,14887,Morty Schapiro,I heard a gunshot and then saw a man run out. ...
1,16371,Annabel Miller,"I saw the murder happen, and I recognized the ..."


In [11]:
print(declaraciones.loc[0]['name'], '\t', declaraciones.loc[0]['transcript'])
print(declaraciones.loc[1]['name'], '\t', declaraciones.loc[1]['transcript'])

Morty Schapiro 	 I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".
Annabel Miller 	 I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


Morty Schapiro:  
- Oye un disparo y un hombre echa a correr
- Socio oro de 'Get Fit Now Gym'
- Núm. de socio comienza con '48Z'
- Coche con matrícula que contiene 'H42W' 

Annabel Miller:
- El asesino fue al gimnasio el 9 de enero

Revisamos entradas del gimnasio del día 9 de enero y buscamos el socio oro que comienza por '48Z'

In [12]:
query = '''
select *
    from get_fit_now_check_in as c
        inner join get_fit_now_member as m on c.membership_id = m.id
    where membership_id like '48Z%'
      and check_in_date = '20180109'
'''
sql_query(query)

,membership_id,check_in_date,check_in_time,check_out_time,id,person_id,name,membership_start_date,membership_status
0,48Z7A,20180109,1600,1730,48Z7A,28819,Joe Germuska,20160305,gold
1,48Z55,20180109,1530,1700,48Z55,67318,Jeremy Bowers,20160101,gold


Hay dos personas que fueron al gimnasio el 9 de enero, revisamos la tabla de licencias de coche para ver de quién puede ser el coche

In [13]:
query = '''
select * 
    from drivers_license as dl
        inner join person as p on p.license_id = dl.id
    where plate_number like '%H42W%'
      and gender = 'male'
'''
sql_query(query)

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model,id,name,license_id,address_number,address_street_name,ssn
0,664760,21,71,black,black,male,4H42WR,Nissan,Altima,51739,Tushar Chandra,664760,312,Phi St,137882671
1,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279


Tenemos 2 sospechosos, Jeremy Bowers y Joe Germuska.  
Todo indica que Jeremy Bowers es nuestro asesino y el coche en el que huyó es suyo.  
Comprobamos si es nuestro asesino.

In [14]:
query = '''
insert into solution
values (1, 'Jeremy Bowers')
'''
crsr.execute(query)


query = f'''
select value
from solution;
'''
sol = sql_query(query)
print(*sol['value'])

Congrats, you found the murderer! But wait, there's more... If you think you're up for a challenge, try querying the interview transcript of the murderer to find the real villain behind this crime. If you feel especially confident in your SQL skills, try to complete this final step with no more than 2 queries. Use this same INSERT statement with your new suspect to check your answer.


JEREMY BOWERS
Congrats, you found the murderer! But wait, there's more... If you think you're up for a challenge, try querying the interview transcript of the murderer to find the real villain behind this crime. If you feel especially confident in your SQL skills, try to complete this final step with no more than 2 queries. Use this same INSERT statement with your new suspect to check your answer.

Indaguemos más. Vamos a ver si el día del asesinato Jeremy estuvo en algún evento.

In [15]:
query = '''
select p.name, fe.*
    from facebook_event_checkin fe
      inner join person p on p.id = fe.person_id
    where p.id = 67318
      and fe.date = '20180115'
'''
sql_query(query)


,name,person_id,event_id,event_name,date
0,Jeremy Bowers,67318,4719,The Funky Grooves Tour,20180115


Jeremy estuvo en el evento The Funky Grooves Tour.  
Veamos las declaraciones de los sospechosos

In [16]:
query = '''
select i.*, p.name
    from interview i
        inner join person p on id = person_id
    where p.name in ('Joe Germuska', 'Jeremy Bowers')
'''
jeremy_declar = sql_query(query)
print(jeremy_declar.loc[0]['name'], '\t', jeremy_declar.loc[0]['transcript'])


Jeremy Bowers 	 I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.



Jeremy Bowers confiesa que la contrato una mujer:
- Tiene mucho dinero
- Altura entre 5'5" (65") y 5'7" (67")
- Pelirroja
- Conduce un Tesla Model S
- Estuvo 3 veces en el SQL Symphony Concert en diciembre de 2017

Revisamos licencias para localizar a la mujer pelirroja. Intentamos localizarla en 2 queries.

In [17]:
query = '''
select p.name, p.id as person_id, dl.car_make, dl.car_model, count(fe.event_id), fe.event_name
    from drivers_license as dl
      inner join person as p on p.license_id = dl.id
      inner join facebook_event_checkin fe on fe.person_id = p.id
    where dl.car_make = 'Tesla'
      and dl.car_model = 'Model S'
      and dl.height between 65 and 67
      and dl.hair_color = 'red'
      and dl.gender = 'female'
      and fe.event_name = 'SQL Symphony Concert'
      and fe.date like '201712%'
    group by p.name
    having count(fe.event_id) = 3
'''
mujer_rica_df = sql_query(query)
mujer_rica_id = mujer_rica_df['person_id'][0]
mujer_rica_id

np.int64(99716)

La única mujer es Miranda Priestly con id = 99716, comprobamos si es rica

In [18]:
query = f'''
select p.id, p.name, i.annual_income, (select avg(annual_income) from income)
    from person as p
        join income as i on p.ssn = i.ssn
    where id =  99716;
'''
sql_query(query)

,id,name,annual_income,(select avg(annual_income) from income)
0,99716,Miranda Priestly,310000,53257.798776


Miranda Priestly gana 310,000$ al año, siendo el sueldo medio 53,257$, confirmamos que es la mujer que menciona Jeremy Bowers. Lo que nos hace sospechar que se trata del cerebro del crimen. Comprobamos:

In [19]:
query = '''
insert into solution
values (1, 'Miranda Priestly')
'''
crsr.execute(query)


query = f'''
select value
from solution;
'''
sol = sql_query(query)
print(*sol['value'])

Congrats, you found the brains behind the murder! Everyone in SQL City hails you as the greatest SQL detective of all time. Time to break out the champagne!


MIRANDA PRIESTLY  
Congrats, you found the brains behind the murder! Everyone in SQL City hails you as the greatest SQL detective of all time. Time to break out the champagne!